In [0]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
repo_root = next(
    (
        path for path in (current, *current.parents)
        if (path / "transformations" / "silver").is_dir()
        and (path / "sources" / "reference").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("No se encontró la raíz del repositorio.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pathlib import Path
import transformations.silver.product_mapping_transform as mapping_module

mapping_file = str(
    (repo_root / "sources/reference/product_external_mapping.csv").resolve()
)

mapping = mapping_module.read_product_mapping(spark, mapping_file)

local_products = spark.table(
    "dbw_arlp_dev_diego01.silver_techretail.products"
)

external_products = spark.table(
    "dbw_arlp_dev_diego01.silver_external.dummyjson_products"
)

checks = mapping_module.validate_product_mapping(
    mapping,
    local_products,
    external_products,
)

mapping_count = mapping.count()
assert mapping_count == 6, "Se esperaban seis asociaciones"

print("CONTROLES:", checks)
display(mapping.orderBy("product_id"))

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

processing_run_id = str(uuid4())
processed_at = datetime.now(timezone.utc)

silver_root = (
    "abfss://silver@starlpdevdiego01.dfs.core.windows.net"
)

prepared_mapping = mapping_module.add_mapping_provenance(
    mapping,
    mapping_file,
    processing_run_id,
    processed_at,
)

result = mapping_module.write_product_mapping_delta(
    spark,
    prepared_mapping,
    silver_root,
)

print("RESULTADO:", result)
print("Processing run ID:", processing_run_id)

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    dbw_arlp_dev_diego01.silver_reference
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS
    dbw_arlp_dev_diego01.silver_reference.product_external_mapping
    USING DELTA
    LOCATION 'abfss://silver@starlpdevdiego01.dfs.core.windows.net/reference/product_external_mapping'
""")

spark.sql("""
    SELECT COUNT(*) AS total_mappings
    FROM dbw_arlp_dev_diego01.silver_reference.product_external_mapping
""").show()